# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library. All dataset entities (record sets, fields, columns, etc.) are referenced by their Croissant schema `@id`.

### Dataset Source
The dataset is defined by a Croissant schema at:
[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Make sure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the FAIR² dataset metadata and display basic description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
md = dataset.metadata

print(f"Dataset: {md.name}\n\nDescription: {md.description}")

## 2. Data Overview
List available RecordSets and Fields. All entities are referenced by their `@id`.

In [ ]:
# Display all record sets by @id

print("Available RecordSets:")
for record_set in dataset.metadata.record_sets:
    print(f"  - {record_set['@id']}")

# Display fields for each RecordSet
for record_set in dataset.metadata.record_sets:
    print(f"\nFields for RecordSet '{record_set['@id']}':")
    for field in record_set['fields']:
        # Print @id, name, dataType
        print(f"    - @id: {field['@id']} | name: {field.get('name', '<no name>')} | type: {field.get('dataType', 'unknown')}")

## 3. Data Extraction
Load the main data record set into a DataFrame. Use the corresponding RecordSet and Field `@id`s.

In [ ]:
# --- Find the primary data record set ---
# Use dataset.metadata.record_sets and pick the tabular data record set (likely the main one)

# For this dataset, the main record set @id is likely:
# 'https://sen.science/doi/10.71728/senscience.qs2f-h81p#RecordSet'
# Let's list all @ids and load the largest one (most fields)

record_sets = dataset.metadata.record_sets

if not record_sets:
    raise ValueError('No RecordSets found in metadata.')

# Pick main record set (greatest number of fields) as it likely holds the data table
main_record_set = max(record_sets, key=lambda rs: len(rs['fields']))
main_record_set_id = main_record_set['@id']

print(f"Main data RecordSet @id: {main_record_set_id}")

# Also, collect all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for rset_id in record_set_ids:
    records = list(dataset.records(record_set=rset_id))
    dataframes[rset_id] = pd.DataFrame(records)

# Show columns (field @ids) for the main table
print(f"Fields (@id) in main RecordSet '{main_record_set_id}':")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Process main data for analysis. Use field `@id` values for selections and transformations.

In [ ]:
# --- Identify numeric and categorical fields by field @id ---
import numpy as np

df = dataframes[main_record_set_id]

# Try to infer a numeric field from the field list
numeric_field = None
# Search for likely numeric fields by column name or dtype
numerics = df.select_dtypes(include=[np.number]).columns.tolist()
if numerics:
    numeric_field = numerics[0]
else:
    # Fallback: try field @ids containing 'age' or 'interval', as plausible numeric
    candidates = [c for c in df.columns if ('age' in c.lower() or 'interval' in c.lower() or 'duration' in c.lower())]
    numeric_field = candidates[0] if candidates else df.columns[0]

print(f"Using numeric field (by @id): {numeric_field}")

# Set threshold for filtering (example: mean+std/2)
try:
    thresh_val = df[numeric_field].astype(float).mean() + df[numeric_field].astype(float).std()/2
    threshold = thresh_val
except Exception:
    threshold = 10  # generic fallback

# Remove outliers above 3 std devs
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    upper = df[numeric_field].mean() + 3*df[numeric_field].std()
    df_eda = df[df[numeric_field] <= upper].copy()
else:
    # Try converting
    df_eda = df.copy()
    try:
        df_eda[numeric_field] = pd.to_numeric(df_eda[numeric_field], errors='coerce')
        upper = df_eda[numeric_field].mean() + 3*df_eda[numeric_field].std()
        df_eda = df_eda[df_eda[numeric_field] <= upper]
    except:
        pass

filtered_df = df_eda[df_eda[numeric_field] > threshold].copy()
print(f"Filtered records where '{numeric_field}' > {threshold}:")
display(filtered_df[[numeric_field]].head())

# Normalize the numeric field (z-score)
filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized: {numeric_field}_normalized")
display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

# Try to find a categorical/group field
cat_candidates = [c for c in df.columns if (('sex' in c.lower()) or ('msi' in c.lower()) or ('group' in c.lower()) or ('anatom' in c.lower())) and c != numeric_field]
group_field = cat_candidates[0] if cat_candidates else None
print(f"Grouping by field (by @id): {group_field}")

# Grouped aggregate (mean)
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nMean {numeric_field} grouped by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Plot distribution and group comparison for the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot of numeric field
plt.figure(figsize=(7,4))
sns.histplot(df_eda[numeric_field].dropna(), kde=True, bins=15, color='royalblue')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

if group_field:
    plt.figure(figsize=(7,5))
    sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field], palette='Set2')
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion

This notebook demonstrated loading, inspecting, and performing elementary processing on the FAIR² dataset using its Croissant schema and the `mlcroissant` library. All operations referenced fields and record sets by their `@id`, supporting reproducible, FAIR-compliant data workflows.

Further steps could include more advanced feature engineering, deeper visualization, and developing ML pipelines for MSI-H prediction or anatomical subgroup analysis using the dataset.